# M1: Cloud Free-CPU Web Grounding Data Prep (0 CCU)
- **IMPORTANT**: Make sure Colab Runtime is **CPU (None accelerator)** so **0 CCU** are spent!
- Streams `agentsea/wave-ui` (Web + Desktop subsets) and `lmms-lab/ScreenSpot-v2`, filters out full-page long screenshots, validates `[0, 1000]` boxes, injects ~5% `not_found` negatives, splits leak-free by `image_hash`, and packs a fast `.tar` bundle into Google Drive.


In [ ]:
import os, shutil, subprocess
from google.colab import drive

if shutil.which("nvidia-smi") is not None:
    gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if gpu_check.returncode == 0:
        raise RuntimeError("GPU runtime detected! Switch Runtime -> Change runtime type -> CPU so you do not burn CCUs.")
print("[OK] CPU-only runtime verified (0 CCU burn).")

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/chrome-gui-pilot"
DATA_DRIVE_DIR = f"{DRIVE_ROOT}/data"
os.makedirs(DATA_DRIVE_DIR, exist_ok=True)
print("Drive target:", DATA_DRIVE_DIR)


In [ ]:
!pip install -q -U "datasets>=3.0" pillow pyarrow tqdm


In [ ]:
import hashlib
import io
import json
import random
from collections import Counter
from pathlib import Path

from datasets import load_dataset
from PIL import Image

MAX_IMAGE_TOKENS = 1024
GRID = 32
MAX_PIXELS = MAX_IMAGE_TOKENS * GRID * GRID  # 1,048,576 pixels (~1280x800)
TARGET_WEB = 16000
TARGET_DESKTOP = 4000
NEG_RATIO = 0.05
SEED = 42

WORK_DIR = Path("/content/m1_build")
IMG_DIR = WORK_DIR / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = (
    "You are a Chrome browser GUI action assistant. Given a Chrome window screenshot and a user instruction, "
    "output strictly one JSON object for the next action using [0, 1000] normalized integer coordinates. "
    "Do not output explanations or markdown."
)

PROMPT_TEMPLATES = [
    "{inst}",
    "Click on: {inst}",
    "Locate and click: {inst}",
    "请点击界面中的：{inst}",
    "帮我找到并点击：{inst}",
]


def resize_for_qwen35(img: Image.Image) -> Image.Image:
    w, h = img.size
    if w * h > MAX_PIXELS:
        scale = (MAX_PIXELS / (w * h)) ** 0.5
        w = max(GRID, int(w * scale))
        h = max(GRID, int(h * scale))
    new_w = max(GRID, round(w / GRID) * GRID)
    new_h = max(GRID, round(h / GRID) * GRID)
    while new_w * new_h > MAX_PIXELS and new_w > GRID and new_h > GRID:
        if new_w >= new_h:
            new_w -= GRID
        else:
            new_h -= GRID
    if (new_w, new_h) != img.size:
        return img.resize((new_w, new_h), Image.Resampling.BICUBIC)
    return img


def is_valid_viewport_image(img: Image.Image) -> bool:
    w, h = img.size
    if w < 320 or h < 240:
        return False
    # Reject extreme full-page scrolling screenshots
    if h / w > 1.85 or w / h > 3.2 or max(w, h) > 4096:
        return False
    return True


def to_norm_box_xyxy(bbox, img_w: int, img_h: int):
    if not bbox or len(bbox) != 4:
        return None
    x1, y1, x2, y2 = [float(v) for v in bbox]
    if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 1.05:
        x1, x2 = x1 * img_w, x2 * img_w
        y1, y2 = y1 * img_h, y2 * img_h
    if x2 <= x1 or y2 <= y1 or x2 > img_w * 1.05 or y2 > img_h * 1.05:
        return None
    nx1 = min(max(round(x1 / max(1, img_w - 1) * 1000.0), 0), 1000)
    ny1 = min(max(round(y1 / max(1, img_h - 1) * 1000.0), 0), 1000)
    nx2 = min(max(round(x2 / max(1, img_w - 1) * 1000.0), 0), 1000)
    ny2 = min(max(round(y2 / max(1, img_h - 1) * 1000.0), 0), 1000)
    if nx2 - nx1 < 2 or ny2 - ny1 < 2:
        return None
    return [nx1, ny1, nx2, ny2]


def make_sft_record(sample_id: str, rel_img_path: str, instruction_text: str, assistant_json: dict) -> dict:
    return {
        "sample_id": sample_id,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"<image>{instruction_text}"},
            {"role": "assistant", "content": json.dumps(assistant_json, ensure_ascii=False)},
        ],
        "images": [rel_img_path],
    }


rng = random.Random(SEED)
counts = Counter()
seen_image_hashes = {}
image_instructions = {}
records = []

print("Streaming agentsea/wave-ui ...")
ds_wave = load_dataset("agentsea/wave-ui", split="train", streaming=True)

for idx, row in enumerate(ds_wave):
    if counts["web"] >= TARGET_WEB and counts["desktop"] >= TARGET_DESKTOP:
        break
    platform = (row.get("platform") or "").lower().strip()
    if platform not in ("web", "desktop"):
        continue
    if platform == "web" and counts["web"] >= TARGET_WEB:
        continue
    if platform == "desktop" and counts["desktop"] >= TARGET_DESKTOP:
        continue

    img = row.get("image")
    if img is None:
        counts["skip_no_img"] += 1
        continue
    try:
        if img.mode != "RGB":
            img = img.convert("RGB")
    except Exception:
        counts["skip_bad_img"] += 1
        continue

    if not is_valid_viewport_image(img):
        counts["skip_aspect_or_size"] += 1
        continue

    orig_w, orig_h = img.size
    res = row.get("resolution")
    ref_w, ref_h = (int(res[0]), int(res[1])) if (res and len(res) == 2 and res[0] > 0 and res[1] > 0) else (orig_w, orig_h)
    norm_box = to_norm_box_xyxy(row.get("bbox"), ref_w, ref_h)
    if norm_box is None and (ref_w, ref_h) != (orig_w, orig_h):
        norm_box = to_norm_box_xyxy(row.get("bbox"), orig_w, orig_h)
    if norm_box is None:
        counts["skip_bad_box"] += 1
        continue

    raw_inst = (row.get("instruction") or row.get("name") or row.get("description") or "").strip()
    if len(raw_inst) < 2 or len(raw_inst) > 180:
        counts["skip_bad_inst"] += 1
        continue

    resized = resize_for_qwen35(img)
    buf = io.BytesIO()
    resized.save(buf, format="JPEG", quality=92)
    img_bytes = buf.getvalue()
    img_hash = hashlib.sha1(img_bytes).hexdigest()[:16]

    rel_path = f"images/wave_{img_hash}.jpg"
    abs_path = WORK_DIR / rel_path
    if img_hash not in seen_image_hashes:
        abs_path.write_bytes(img_bytes)
        seen_image_hashes[img_hash] = (rel_path, resized.size[0], resized.size[1])
        image_instructions[img_hash] = set()
    image_instructions[img_hash].add(raw_inst.lower())

    cx = (norm_box[0] + norm_box[2]) // 2
    cy = (norm_box[1] + norm_box[3]) // 2
    prompt_text = rng.choice(PROMPT_TEMPLATES).format(inst=raw_inst)
    sample_id = f"wave_{platform}_{idx:07d}"

    sft_rec = make_sft_record(sample_id, rel_path, prompt_text, {"action": "left_click", "coordinate": [cx, cy]})
    meta_rec = {
        "sample_id": sample_id,
        "image_path": rel_path,
        "instruction": prompt_text,
        "raw_instruction": raw_inst,
        "action": "left_click",
        "coordinate": [cx, cy],
        "bbox": norm_box,
        "source": f"waveui_{row.get('source') or platform}",
        "platform": platform,
        "site_domain": str(row.get("source") or "unknown"),
        "session_id": img_hash,
        "crop_mode": "upstream_dataset",
        "img_w": resized.size[0],
        "img_h": resized.size[1],
    }
    records.append((img_hash, sft_rec, meta_rec))
    counts[platform] += 1
    if (counts["web"] + counts["desktop"]) % 2000 == 0:
        print(f"Collected web={counts['web']}, desktop={counts['desktop']}, unique_imgs={len(seen_image_hashes)}")

print("WaveUI collection summary:", dict(counts))


In [ ]:
# Add ~5% negative (not_found) samples using mismatched cross-screenshot instructions
all_hashes = list(seen_image_hashes.keys())
all_insts = [(h, list(insts)[0]) for h, insts in image_instructions.items() if insts]
num_neg = int(len(records) * NEG_RATIO)
neg_added = 0
for _ in range(num_neg * 3):
    if neg_added >= num_neg:
        break
    h_img = rng.choice(all_hashes)
    h_other, neg_inst = rng.choice(all_insts)
    if h_other == h_img or neg_inst in image_instructions[h_img]:
        continue
    rel_path, rw, rh = seen_image_hashes[h_img]
    prompt_text = rng.choice(PROMPT_TEMPLATES).format(inst=neg_inst)
    sample_id = f"wave_neg_{neg_added:06d}"
    sft_rec = make_sft_record(
        sample_id,
        rel_path,
        prompt_text,
        {"action": "not_found", "reason": "Target element is not present in the current screenshot"},
    )
    meta_rec = {
        "sample_id": sample_id,
        "image_path": rel_path,
        "instruction": prompt_text,
        "raw_instruction": neg_inst,
        "action": "not_found",
        "coordinate": None,
        "bbox": None,
        "source": "waveui_synthetic_neg",
        "platform": "web",
        "site_domain": "synthetic_neg",
        "session_id": h_img,
        "crop_mode": "upstream_dataset",
        "img_w": rw,
        "img_h": rh,
    }
    records.append((h_img, sft_rec, meta_rec))
    neg_added += 1

print(f"Added {neg_added} synthetic negative samples.")

# Leak-free train/val split strictly grouped by image hash
unique_hashes = sorted(seen_image_hashes.keys())
rng.shuffle(unique_hashes)
val_cut = max(1, int(len(unique_hashes) * 0.05))
val_hash_set = set(unique_hashes[:val_cut])

train_sft, val_sft, meta_all = [], [], []
for h, sft_rec, meta_rec in records:
    meta_all.append(meta_rec)
    if h in val_hash_set:
        val_sft.append(sft_rec)
    else:
        train_sft.append(sft_rec)

rng.shuffle(train_sft)
print(f"Split by image_hash: train={len(train_sft)}, val={len(val_sft)} (zero shared screenshots)")


In [ ]:
# Prepare ScreenSpot-v2 evaluation split (verified [x, y, w, h] pixel format)
print("Loading lmms-lab/ScreenSpot-v2 ...")
ss_ds = load_dataset("lmms-lab/ScreenSpot-v2", split="train")
ss_sft, ss_meta = [], []
ss_sources = Counter()

for idx, row in enumerate(ss_ds):
    img = row["image"]
    if img.mode != "RGB":
        img = img.convert("RGB")
    orig_w, orig_h = img.size
    bx, by, bw, bh = [float(v) for v in row["bbox"]]
    norm_box = to_norm_box_xyxy([bx, by, bx + bw, by + bh], orig_w, orig_h)
    if norm_box is None:
        continue

    resized = resize_for_qwen35(img)
    rel_path = f"images/screenspot_{idx:05d}.jpg"
    resized.save(WORK_DIR / rel_path, format="JPEG", quality=95)

    cx = (norm_box[0] + norm_box[2]) // 2
    cy = (norm_box[1] + norm_box[3]) // 2
    inst = row["instruction"].strip()
    ds_src = str(row.get("data_source") or "unknown").lower()
    ds_type = str(row.get("data_type") or "unknown").lower()
    ss_sources[ds_src] += 1

    sample_id = f"ssv2_{idx:05d}"
    ss_sft.append(make_sft_record(sample_id, rel_path, inst, {"action": "left_click", "coordinate": [cx, cy]}))
    ss_meta.append({
        "sample_id": sample_id,
        "image_path": rel_path,
        "instruction": inst,
        "action": "left_click",
        "coordinate": [cx, cy],
        "bbox": norm_box,
        "source": f"screenspot_v2_{ds_src}",
        "data_type": ds_type,
        "data_source": ds_src,
        "img_w": resized.size[0],
        "img_h": resized.size[1],
    })

print("ScreenSpot-v2 loaded:", len(ss_sft), dict(ss_sources))


In [ ]:
import os
import shutil
import tarfile

def write_jsonl(path: Path, rows: list[dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(WORK_DIR / "train_open_web.jsonl", train_sft)
write_jsonl(WORK_DIR / "val_open_web.jsonl", val_sft)
write_jsonl(WORK_DIR / "metadata_open_web.jsonl", meta_all)
write_jsonl(WORK_DIR / "val_screenspot_v2.jsonl", ss_sft)
write_jsonl(WORK_DIR / "metadata_screenspot_v2.jsonl", ss_meta)

stats = {
    "max_image_tokens": MAX_IMAGE_TOKENS,
    "train_count": len(train_sft),
    "val_open_count": len(val_sft),
    "screenspot_v2_count": len(ss_sft),
    "wave_counts": dict(counts),
    "screenspot_sources": dict(ss_sources),
    "neg_samples": neg_added,
}
(WORK_DIR / "stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")

tar_local = Path("/content/m1_web_bundle.tar")
print("Packing tar archive...")
with tarfile.open(tar_local, "w") as tar:
    tar.add(WORK_DIR, arcname=".")

size_mb = tar_local.stat().st_size / (1024**2)
DATA_DRIVE_DIR = "/content/drive/MyDrive/chrome-gui-pilot/data"
os.makedirs(DATA_DRIVE_DIR, exist_ok=True)
print(f"Archive size: {size_mb:.1f} MiB -> copying to Google Drive...")
shutil.copy2(tar_local, Path(DATA_DRIVE_DIR) / "m1_web_bundle.tar")
shutil.copy2(WORK_DIR / "stats.json", Path(DATA_DRIVE_DIR) / "m1_stats.json")
print("[DONE] Data bundle saved to Drive:", f"{DATA_DRIVE_DIR}/m1_web_bundle.tar")
